In [ ]:
import ast
import os
import ipynbname
import pandas as pd
import numpy as np
from Functions.AutoCloud_V4_5 import *
from Functions.Utils import *
from Functions.Graphs import *
from Functions.TedaGraphs import *
from Functions.TedaOptimize_V4_5 import Optimize, OptimizeRho, RunTag
import copy

FileName = ipynbname.name()

# Configuração do experimento
FEEDBACK = 'symmetric'      # 'symmetric' (gradiente no passo atual) ou 'random' (ablação: RFLO + DFA)
WTAG, WTAP = False, False   # configuração principal: soma ponderada dos grânulos (eq. 36)
EOL = 0.3                   # limiar de fim de vida (EoL) do HI
W = 1                       # janela do DSI em cada X_k (nº de amostras do RS; PrepareData/WindowDSI)

LAM = 1.0                   # fator de esquecimento (1.0 = todos os passos com o mesmo peso)

out_path = os.path.join('Optimization', RunTag(FileName, FEEDBACK, WTAG, WTAP), 'multi', 'Optimization.csv')
dfs = [pd.read_csv(out_path)] if os.path.exists(out_path) else []
RS = pd.read_excel(r'Dataset\RS.xlsx')
HI = pd.read_excel(r'Dataset\HI.xlsx')
sig = HI['PC1'].values

In [ ]:
params = [3.5,28,[70],8,0,1,2,5,19,0]        # [m, nI, nR, nO (futuros de Z_k), mO (passados de Z_k, incl. h_k), N1, N2, N3, τ, act]
m,nI,nR,nO,mO,N1,N2,N3,τ,act = ParseParams(params)
X, Y, Z, kX = PrepareData(RS, HI, W, nI, nO, mO)
ref, J, L = RefEoL(sig, EOL, kX)   # avisa se o fluxo não alcança o EoL
if isinstance(nR, str): nR = ast.literal_eval(nR)


# RUL intervalar (RULInterv2): protocolo 3 na janela de entrada + análise intervalar da rede (pesos numéricos)
RHO = 0.76                  # rho inicial: múltiplo do erro estimado do HI (eP) somado/subtraído aos valores previstos
# rho dinâmico (sugestão 3): subida de gradiente online em V = média(cov*sp) do HI
RHO_DYN = True              # False: rho estático (= RHO)
RHO_ETA, RHO_T, RHO_BETA = 0.51, 0.51, 0.59   # passo, temperatura da cobertura suavizada (escore |e|/e_est), momento
RHO_MIN, RHO_MAX = 1e-2, 10
INTR_STATE = 'reset'        # estado recorrente intervalar: 'reset' (só a incerteza da janela do passo) ou 'carry' (propagado)

teda=AutoCloud(m=m,nI=nI,nR=nR,nO=nO,ηS=[N1,N2,N3],act=act,
               tau=τ,eol=EOL,ref=ref,wtaG=WTAG,wtaP=WTAP,feedback=FEEDBACK,mO=mO,
               rho=RHO,lam=LAM,intr_state=INTR_STATE,
               rho_dyn=RHO_DYN,rho_eta=RHO_ETA,rho_T=RHO_T,rho_beta=RHO_BETA,rho_min=RHO_MIN,rho_max=RHO_MAX,
               ff_rls=0.91,dt_rls=27
               )

for j,_ in enumerate(X[:J]):

    teda.run(X[j])
    teda.RUL_Prediction(Y[j],mode='interval',lim=L,show=False)
    teda.WAPE_HI2(Y[j],Z[j])
    teda.Adapt(Y[j],Z[j])
    teda.UpdateRLS(mode='HI')

print(f"HI (HI_Granularity): cov={teda.cov:.3f} | sp={teda.sp:.3f} | V=média(cov*sp)={teda.V:.4f}")
PlotDSI_3D_PLY(teda,xi=10, yf=100)


In [ ]:
print(df_rho)

In [ ]:
lS,lE = 1,2
for nL in range(lS,lE+1):
    df = Optimize(RS=RS, HI=HI, eol=EOL, w=W, lam=LAM,
                  FileName=FileName, OptDim=1, OptSampler='tpe', OptSeed=100*nL, OptPrune=False,
                  n_study=2, n_trials=3e3, timeout=1200, patience=1000,
                  mS=[2.0,4.5], nLS=[nL,nL], nRS=[1,70], nOS=[1,9], mOS=[0,10], actS=[0,1],
                  feedback=FEEDBACK, wtaG=WTAG, wtaP=WTAP, OptReset=False)

In [13]:
params = df.iloc[9,1:].values
m,nI,nR,nO,mO,N1,N2,N3,τ,act = ParseParams(params)
X, Y, Z, kX = PrepareData(RS, HI, W, nI, nO, mO)
ref, J, L = RefEoL(sig, EOL, kX)   # avisa se o fluxo não alcança o EoL
if isinstance(nR, str): nR = ast.literal_eval(nR)


# RUL intervalar (RULInterv2): protocolo 3 na janela de entrada + análise intervalar da rede (pesos numéricos)
RHO = 0.76                  # rho inicial: múltiplo do erro estimado do HI (eP) somado/subtraído aos valores previstos
# rho dinâmico (sugestão 3): subida de gradiente online em V = média(cov*sp) do HI
RHO_DYN = True              # False: rho estático (= RHO)
RHO_ETA, RHO_T, RHO_BETA = 0.51, 0.51, 0.59   # passo, temperatura da cobertura suavizada (escore |e|/e_est), momento
RHO_MIN, RHO_MAX = 1e-2, 10
INTR_STATE = 'reset'        # estado recorrente intervalar: 'reset' (só a incerteza da janela do passo) ou 'carry' (propagado)

teda=AutoCloud(m=m,nI=nI,nR=nR,nO=nO,ηS=[N1,N2,N3],act=act,
               tau=τ,eol=EOL,ref=ref,wtaG=WTAG,wtaP=WTAP,feedback=FEEDBACK,mO=mO,
               rho=RHO,lam=LAM,intr_state=INTR_STATE,
               rho_dyn=RHO_DYN,rho_eta=RHO_ETA,rho_T=RHO_T,rho_beta=RHO_BETA,rho_min=RHO_MIN,rho_max=RHO_MAX,
               ff_rls=0.91,dt_rls=27
               )

for j,_ in enumerate(X[:J]):

    teda.run(X[j])
    teda.RUL_Prediction(Y[j],mode='interval',lim=L,show=False)
    teda.WAPE_HI2(Y[j],Z[j])
    teda.Adapt(Y[j],Z[j])
    teda.UpdateRLS(mode='HI')

print(f"HI (HI_Granularity): cov={teda.cov:.3f} | sp={teda.sp:.3f} | V=média(cov*sp)={teda.V:.4f}")
PlotDSI_3D_PLY(teda,xi=10, yf=100, HideCol=[1,2])


HI (HI_Granularity): cov=0.757 | sp=0.870 | V=média(cov*sp)=0.6876


In [ ]:
# Etapa 1: modelo (predições pontuais de HI e RUL, rho fixo)
# df_opt = Optimize(RS=RS, HI=HI, eol=EOL, w=W, lam=LAM, FileName=FileName, OptDim=1, OptSampler='TPE', OptSeed=0, n_study=1, n_trials=500, timeout=3600, nLS=[1,1])
# Etapa 2: rho e parâmetros do rho dinâmico (maximiza V) no melhor modelo da etapa 1 (ou params=[...])
# df_rho = OptimizeRho(RS=RS, HI=HI, eol=EOL, w=W, lam=LAM, FileName=FileName, params=None, OptSampler='TPE', OptSeed=0, n_trials=100, timeout=3600, Vtarget='HI')